# Confluence v2 — getting started

Computational closed loop: noisy cancer observations drive a fruit-fly mushroom-body-style network, which writes a five-drug infusion vector into a mechanistic microenvironment ODE.

**This notebook is a simulation walkthrough. It is not a clinical tool and does not claim therapeutic benefit.**

Interactive UI (preferred):

```bash
python -m confluence
```

In [ ]:
from confluence.cancer_env.archetypes import get_archetype, ARCHETYPES
from confluence.cancer_env.ode_system import CancerODE
from confluence.controllers import make_controller
from confluence.loop import ClosedLoopSimulator
from confluence.pharmacology.toxicity_constraints import load_drug_catalog

print("archetypes:", list(ARCHETYPES))
print("catalog:", [d.id for d in load_drug_catalog()])

## Open-loop ODE (no controller)

Integrate glioblastoma with a constant modest infusion and confirm the 11-D state stays finite.

In [ ]:
import numpy as np

ode = CancerODE(get_archetype("glioblastoma"))
x, c = ode.initial_state()
u = np.array([0.2, 0.1, 0.15, 0.1, 0.3])
xs = []
for _ in range(200):
    x, c = ode.step(x, c, u, dt=0.2)
    xs.append(x.copy())
xs = np.array(xs)
print("finite", np.all(np.isfinite(xs)), "final H", float(xs[-1, 10]), "burden", float(xs[-1, 0] + xs[-1, 1]))

## Closed loop with controller E (plastic mushroom body)

256 Kenyon cells is the interactive default. Observations flow into the network; dopamine updates KC→MBON weights.

In [ ]:
sim = ClosedLoopSimulator(
    archetype="melanoma_persister",
    controller=make_controller("E", n_kc=128, seed=3),
    dt=0.25,
    seed=3,
)
burdens, das, us = [], [], []
for _ in range(80):
    fr = sim.step()
    burdens.append(fr.latent.tumor_burden)
    das.append((fr.connectome or {}).get("da", 0.0))
    us.append(sum(fr.action.infusion.values()))
print("steps", len(burdens), "final burden", burdens[-1], "mean U", float(np.mean(us)), "mean DA", float(np.mean(das)))

## Compare A vs B vs E on PDAC (short horizon)

For a fuller bake-off: `python -m confluence --benchmark --trials 2`.

In [ ]:
from confluence.benchmarks.runner import run_trial

for letter in ("A", "B", "E"):
    m = run_trial("pancreatic_pdac", letter, seed=4, horizon_days=25, dt=0.5, n_kc=64)
    print(letter, "PFS", round(m.pfs_days, 1), "resist_t", m.resistance_emergence_day, "tox", round(m.cumulative_toxicity, 3))